# Customer CSV analysis

This notebook answers two questions:

1. Profile `customers-100000.csv` and summarize useful customer-data insights.
2. Process `customers-2000000.csv` with low memory usage by reading it in chunks instead of loading it all at once.

The large-file workflow keeps only aggregate counters and a small sample in memory.

In [1]:
from collections import Counter
from pathlib import Path
import os

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "dataset").exists():
    ROOT = Path("..").resolve()

SMALL_PATH = ROOT / "dataset" / "customers-100000.csv"
LARGE_PATH = ROOT / "dataset" / "customers-2000000" / "customers-2000000.csv"
TEXT_COLUMNS = ["Customer Id", "First Name", "Last Name", "Company", "City", "Country", "Phone 1", "Phone 2", "Email", "Website"]
DTYPES = {"Index": "int32", **{column: "string" for column in TEXT_COLUMNS}}
DATE_COLUMNS = ["Subscription Date"]

print(f"Small file: {SMALL_PATH}")
print(f"Large file: {LARGE_PATH}")

Small file: C:\Users\nanda\Documents\Work\job_prep\Shopee\202610\repo\receipt rag\dataset\customers-100000.csv
Large file: C:\Users\nanda\Documents\Work\job_prep\Shopee\202610\repo\receipt rag\dataset\customers-2000000\customers-2000000.csv


## 1. Profile the 100,000-row file

The smaller file is loaded once for a detailed profile. Explicit dtypes reduce object-column overhead and parsing the date column makes time-based insights reliable.

In [2]:
small = pd.read_csv(
    SMALL_PATH,
    dtype={**DTYPES, "Subscription Date": "string"},
    parse_dates=DATE_COLUMNS,
)

print(f"Rows: {len(small):,}")
print(f"Columns: {small.shape[1]}")
print(f"DataFrame memory: {small.memory_usage(deep=True).sum() / 1024**2:.1f} MiB")
display(small.head())
display(small.dtypes.to_frame("dtype"))

def missing_report(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.isna().sum().rename("missing").to_frame()
    result["missing_pct"] = (result["missing"] / len(frame) * 100).round(2)
    return result.sort_values(["missing", "missing_pct"], ascending=False)

def duplicate_report(frame: pd.DataFrame) -> dict[str, int]:
    return {
        "duplicate_rows": int(frame.duplicated().sum()),
        "duplicate_customer_ids": int(frame["Customer Id"].duplicated().sum()),
        "duplicate_emails": int(frame["Email"].duplicated().sum()),
    }

display(missing_report(small))
display(pd.Series(duplicate_report(small), name="count"))
small["subscription_year"] = small["Subscription Date"].dt.year
small["email_domain"] = small["Email"].str.rsplit("@", n=1).str[-1].str.lower()
print("Top countries")
display(small["Country"].value_counts().head(10).rename("customers").to_frame())
print("Subscriptions by year")
display(small["subscription_year"].value_counts().sort_index().rename("customers").to_frame())
print("Top email domains")
display(small["email_domain"].value_counts().head(10).rename("customers").to_frame())

Rows: 100,000
Columns: 12
DataFrame memory: 22.6 MiB


,Index,Customer Id,First Name,Last Name,Company,City,Country,Phone 1,Phone 2,Email,Subscription Date,Website
0,1,ffeCAb7AbcB0f07,Jared,Jarvis,Sanchez-Fletcher,Hatfieldshire,Eritrea,274.188.8773x41185,001-215-760-4642x969,gabriellehartman@benjamin.com,2021-11-11,https://www.mccarthy.info/
1,2,b687FfC4F1600eC,Marie,Malone,Mckay PLC,Robertsonburgh,Botswana,283-236-9529,(189)129-8356x63741,kstafford@sexton.com,2021-05-14,http://www.reynolds.com/
2,3,9FF9ACbc69dcF9c,Elijah,Barrera,Marks and Sons,Kimbury,Barbados,8252703789,459-916-7241x0909,jeanettecross@brown.com,2021-03-17,https://neal.com/
3,4,b49edDB1295FF6E,Sheryl,Montgomery,"Kirby, Vaughn and Sanders",Briannaview,Antarctica (the territory South of 60 deg S),425.475.3586,(392)819-9063,thomassierra@barrett.com,2020-09-23,https://www.powell-bryan.com/
4,5,3dcCbFEB17CCf2E,Jeremy,Houston,Lester-Manning,South Brianna,Micronesia,+1-223-666-5313x4530,252-488-3850x692,rubenwatkins@jacobs-wallace.info,2020-09-18,https://www.carrillo.com/


,dtype
Index,int32
Customer Id,string
First Name,string
Last Name,string
Company,string
City,string
Country,string
Phone 1,string
Phone 2,string
Email,string


,missing,missing_pct
Index,0,0.0
Customer Id,0,0.0
First Name,0,0.0
Last Name,0,0.0
Company,0,0.0
City,0,0.0
Country,0,0.0
Phone 1,0,0.0
Phone 2,0,0.0
Email,0,0.0


duplicate_rows            0
duplicate_customer_ids    0
duplicate_emails          5
Name: count, dtype: int64

...

,customers
Country,
Congo,835
Korea,820
Saudi Arabia,463
Pitcairn Islands,456
Saint Martin,453
Paraguay,445
Canada,444
American Samoa,443
Saint Kitts and Nevis,443


Subscriptions by year


,customers
subscription_year,
2020,41898
2021,41211
2022,16891


Top email domains


,customers
email_domain,
mckee.com,59
pugh.com,59
lawrence.com,57
terrell.com,57
gaines.com,56
romero.com,56
delgado.com,56
fitzpatrick.com,55
jacobson.com,55


## 2. Parse the 2-million-row file with low memory

`read_csv(..., chunksize=...)` returns an iterator. Each chunk is processed and released before the next chunk is read. Only aggregate counters and five example rows are retained.

In [3]:
def profile_large_csv(path: Path, chunksize: int = 50_000) -> dict:
    row_count = 0
    missing_counts = Counter()
    country_counts = Counter()
    year_counts = Counter()
    domain_counts = Counter()
    sample_rows = []

    for chunk in pd.read_csv(
        path,
        dtype={**DTYPES, "Subscription Date": "string"},
        chunksize=chunksize,
    ):
        row_count += len(chunk)
        missing_counts.update(chunk.isna().sum().to_dict())
        country_counts.update(chunk["Country"].dropna())
        years = pd.to_datetime(chunk["Subscription Date"], errors="coerce").dt.year.dropna().astype("int16")
        year_counts.update(years.tolist())
        domains = chunk["Email"].dropna().str.rsplit("@", n=1).str[-1].str.lower()
        domain_counts.update(domains)
        if len(sample_rows) < 5:
            sample_rows.extend(chunk.head(5 - len(sample_rows)).to_dict("records"))

    return {
        "rows": row_count,
        "missing": dict(missing_counts),
        "top_countries": country_counts.most_common(10),
        "subscriptions_by_year": sorted(year_counts.items()),
        "top_email_domains": domain_counts.most_common(10),
        "sample": sample_rows,
    }

large_profile = profile_large_csv(LARGE_PATH)
print(f"Rows processed: {large_profile['rows']:,}")
display(pd.DataFrame(large_profile["top_countries"], columns=["country", "customers"]))
display(pd.DataFrame(large_profile["subscriptions_by_year"], columns=["year", "customers"]))
display(pd.DataFrame(large_profile["top_email_domains"], columns=["domain", "customers"]))
display(pd.DataFrame(large_profile["sample"]))

Rows processed: 2,000,000


,country,customers
0,Korea,16240
1,Congo,16208
2,Jordan,8428
3,Vietnam,8388
4,Suriname,8375
5,Timor-Leste,8372
6,Zimbabwe,8360
7,Slovakia (Slovak Republic),8359
8,Netherlands,8354
9,Samoa,8354


,year,customers
0,2020,831045
1,2021,830392
2,2022,338563


,domain,customers
0,rojas.com,898
1,benson.com,889
2,huang.com,886
3,valentine.com,873
4,schmidt.com,870
5,barrera.com,869
6,wolfe.com,869
7,tapia.com,867
8,huynh.com,867
9,le.com,865


,Index,Customer Id,First Name,Last Name,Company,City,Country,Phone 1,Phone 2,Email,Subscription Date,Website
0,1,4962fdbE6Bfee6D,Pam,Sparks,Patel-Deleon,Blakemouth,British Indian Ocean Territory (Chagos Archipe...,267-243-9490x035,480-078-0535x889,nicolas00@faulkner-kramer.com,2020-11-29,https://nelson.com/
1,2,9b12Ae76fdBc9bE,Gina,Rocha,"Acosta, Paul and Barber",East Lynnchester,Costa Rica,027.142.0940,+1-752-593-4777x07171,yfarley@morgan.com,2021-01-03,https://pineda-rogers.biz/
2,3,39edFd2F60C85BC,Kristie,Greer,Ochoa PLC,West Pamela,Ecuador,+1-049-168-7497x5053,+1-311-216-7855,jennyhayden@petty.org,2021-06-20,https://mckinney.com/
3,4,Fa42AE6a9aD39cE,Arthur,Fields,Moyer-Wang,East Belinda,Afghanistan,001-653-754-7486x65787,521-630-3858x953,igrimes@ruiz-todd.org,2020-02-13,https://dominguez.biz/
4,5,F5702Edae925F1D,Michelle,Blevins,Shah and Sons,West Jared,Marshall Islands,8735278329,(633)283-6034x500,diamondcarter@jordan.com,2020-10-20,http://murillo-ryan.com/
